# Lab 3\. Working with Tensors: From Creation to Model-Ready Images

## Setup: Environment Preparation

The steps in this section only prepare the environment. They are not the learning content of the lab, and on a managed course platform some of them may already be done for you. Run them once at the top of your notebook, then move on to Step 1\.

### Setup 1\. Install and Verify PyTorch and TorchVision

Install the two libraries the lab uses. PyTorch provides tensors and the operations on them; TorchVision provides the image dataset loader and the image transforms used at the end of the lab.

In [ ]:
!pip install torch torchvision

Confirm the versions so your results line up with this lab. Printing the versions now also verifies the import works before you depend on it later.

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)

Output:

```
PyTorch: 2.12.1
TorchVision: 0.27.0
```

If your versions are close to these but not identical, the lab still works; the tensor and transform APIs used here are stable across the 2.x line.

### Setup 2\. Download the Rock-Paper-Scissors Dataset

Download the course dataset so the final steps have real images to work with. The images are organized with one folder per class, which is the layout the `ImageFolder` loader expects later in the lab.

In [ ]:
!git clone https://github.com/lftraining/LFD2000.git

After the clone completes, the images are under `LFD2000/datasets/RPS`, with a separate subfolder for each hand pose. You will point the loader at that path in Step 6\.

Output:

```
Cloning into 'LFD2000'...
remote: Enumerating objects: ...
Receiving objects: 100% ...
Resolving deltas: 100% ...
```

## Step 1\. Create Tensors Several Ways

A tensor can come from data you already have or from a factory function that generates values for a shape you specify. You will use every one of these methods later, so start by creating one of each and looking at it.

Create a tensor from a Python list. Nested lists become rows and columns, so a list of two lists of three numbers becomes a two-by-three tensor.

In [ ]:
from_list = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(from_list)

Output:

```
tensor([[1, 2, 3],
        [4, 5, 6]])
```

Now create tensors with factory functions. Each one takes the shape you want. `torch.zeros` and `torch.ones` fill the tensor with `0.0` and `1.0`; `torch.rand` fills it with random values in the range from `0` up to but not including `1`; and `torch.arange(start, stop, step)` counts from `start` in increments of `step`, stopping before `stop`.

In [ ]:
zeros = torch.zeros(2, 3)
ones = torch.ones(2, 3)
randoms = torch.rand(2, 3)
counted = torch.arange(0, 10, 2)

print(zeros)
print(randoms)
print(counted)

Output:

```
tensor([[0., 0., 0.],
        [0., 0., 0.]])
tensor([[0.8213, 0.1050, 0.6420],
        [0.7397, 0.5115, 0.0632]])
tensor([0, 2, 4, 6, 8])
```

The `zeros` and `counted` tensors are deterministic and will match the values above. The `randoms` tensor holds different numbers every run, which is expected; only its two-by-three shape is fixed. \[\]

## Step 2\. Inspect Tensor Properties

Every tensor carries three properties worth checking before you use it: its `shape`, its `dtype`, and its `device`. Reading them is the fastest way to understand a tensor you did not create yourself, and it is the first thing to do when an operation fails.

Print all three for a random tensor.

In [ ]:
t = torch.rand(3, 4)
print("Shape:", t.shape)
print("Dtype:", t.dtype)
print("Device:", t.device)

Output:

```
Shape: torch.Size([3, 4])
Dtype: torch.float32
Device: cpu
```

The shape reports three rows and four columns. The dtype is `torch.float32`, PyTorch's default type for decimal tensors. The device is `cpu`: new tensors are created on the central processor, and they stay there until you explicitly move them, even on a machine that has a GPU. Chapter 4 is where you move tensors to other devices; for now, confirming that a fresh tensor reports `cpu` is the point. \[\]

Check the effect of the values you pass in on dtype. A list of whole numbers produces an integer tensor, while a single decimal point produces a float tensor.

In [ ]:
ints = torch.tensor([1, 2, 3])
floats = torch.tensor([1.0, 2.0, 3.0])
print(ints.dtype, floats.dtype)

Output:

```
torch.int64 torch.float32
```

This is worth seeing directly, because a dtype you did not intend is a common cause of later errors when an operation expects decimals.

## Step 3\. Reshape, Index, Slice, and Combine Tensors

This step covers the manipulations you will use constantly: changing a tensor's shape, selecting parts of it, and combining tensors with arithmetic. Run each block and read the result before continuing.

### Step 3.1. Reshape

Reshaping rearranges the same values into a new shape without changing any of them. Create a flat tensor of twelve values and view it as three rows of four.

In [ ]:
x = torch.arange(12)
y = x.reshape(3, 4)
print(y)

Output:

```
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
```

Passing `-1` for a dimension tells PyTorch to compute that dimension from the total number of elements. You will use `reshape(-1)` at the end of the lab to flatten an image, so try it now.

In [ ]:
flat = y.reshape(-1)
print(flat.shape)

Output:

```
torch.Size([12])
```

### Step 3.2. Index and Slice

Indexing selects part of a tensor with square brackets. A single index selects along the first dimension, a comma separates dimensions, and a colon means every element along that dimension.

In [ ]:
y = torch.arange(12).reshape(3, 4)
print(y[0])       # first row
print(y[:, 1])    # column at index 1, every row
print(y[1, 2])    # single value at row 1, column 2

Output:

```
tensor([0, 1, 2, 3])
tensor([1, 5, 9])
tensor(6)
```

The middle line is the pattern you will reuse on images. For an image tensor shaped as channels by height by width, `image[0]` selects the first color channel in the same way `y[0]` selects the first row here.

### Step 3.3. Arithmetic and Matrix Multiplication

Arithmetic between tensors of the same shape happens element by element. Multiplication with `*` multiplies matching positions, which is different from matrix multiplication.

In [ ]:
a = torch.tensor([1, 2, 3])
b = torch.tensor([10, 20, 30])
print(a + b)
print(a * b)

Output:

```
tensor([11, 22, 33])
tensor([10, 40, 90])
```

Matrix multiplication uses the `@` operator and follows the inner-dimension rule: the number of columns in the first tensor must equal the number of rows in the second. A three-column tensor multiplies a three-row tensor, and the result takes the outer dimensions.

In [ ]:
m1 = torch.rand(2, 3)
m2 = torch.rand(3, 2)
print((m1 @ m2).shape)

Output:

```
torch.Size([2, 2])
```

If the inner dimensions do not match, PyTorch raises an error that names the two shapes, which points you straight at the fix.

### Step 3.4. Broadcasting

Broadcasting combines tensors of different shapes when the shapes are compatible, comparing dimensions from the last one backward. Two dimensions are compatible when they are equal or when one of them is `1`, and a size-`1` dimension is stretched to match. Add a three-element row to every row of a two-by-three tensor.

In [ ]:
matrix = torch.ones(2, 3)
row = torch.tensor([1.0, 2.0, 3.0])
print(matrix + row)

Output:

```
tensor([[2., 3., 4.],
        [2., 3., 4.]])
```

The row lined up with the last dimension of the matrix, so PyTorch added it to each row without a loop. \[Inferred from docs: broadcasting semantics documented in PyTorch, "Broadcasting semantics"; this specific rock-paper-scissors-adjacent example is applied for the lab\]

## Step 4\. Convert Between Tensors and NumPy

PyTorch and NumPy interoperate closely, and much data arrives as NumPy arrays. Converting an array to a tensor with `torch.from_numpy` does not copy the numbers; the tensor and the array share the same memory, so a change to one appears in the other.

In [ ]:
import numpy as np

array = np.array([1.0, 2.0, 3.0])
tensor = torch.from_numpy(array)
print(tensor, tensor.dtype)

Output:

```
tensor([1., 2., 3.], dtype=torch.float64) torch.float64
```

The dtype is `torch.float64`, because NumPy defaults to that higher-precision decimal type and the conversion keeps it. When a model expects `torch.float32`, this is a value to convert deliberately, which the image pipeline does in Step 5\.

Confirm the shared memory by editing the array and reading the tensor.

In [ ]:
array[0] = 99.0
print(tensor)

Output:

```
tensor([99.,  2.,  3.], dtype=torch.float64)
```

The tensor reflects the change because it never held its own copy. Converting the other direction uses the tensor's `.numpy()` method, which returns an array that shares memory the same way.

## Step 5\. Turn One Image into a Tensor

Now apply the tensor skills to the course data, starting with a single image. A model needs every input as a tensor of the same shape and a decimal dtype, so you build a transform pipeline that loads an image, resizes it, and converts it to `torch.float32` with values rescaled into the range from `0.0` to `1.0`.

A transform is a step that takes data in and returns changed data out, and `v2.Compose` chains several into one pipeline. `v2.ToImage` turns a loaded image into a tensor, `v2.Resize` forces a fixed height and width, and `v2.ToDtype` with `scale=True` converts to `float32` and rescales the pixel values from the 0–255 range to the 0.0–1.0 range.

In [ ]:
from torchvision.transforms import v2

to_tensor = v2.Compose([
    v2.ToImage(),
    v2.Resize((128, 128)),
    v2.ToDtype(torch.float32, scale=True),
])

This defines the pipeline; it does not run on anything yet. You apply it through the dataset loader in Step 6\.

**Note:** Older tutorials use a single `v2.ToTensor` transform in place of `v2.ToImage` followed by `v2.ToDtype`. `ToTensor` is deprecated in current TorchVision, so this lab uses the two-step form. \[SME: confirm the course standardizes on `ToImage` \+ `ToDtype` in all chapters and any starter notebooks.\]

## Step 6\. Load the Whole Dataset with ImageFolder

Because the images are arranged one folder per class, `ImageFolder` can load the entire dataset and label each image from its folder name. Point it at the dataset path from the setup section and pass the transform pipeline you just built.

In [ ]:
from torchvision.datasets import ImageFolder

dataset = ImageFolder(root="LFD2000/datasets/RPS", transform=to_tensor)

print("Number of images:", len(dataset))
print("Classes:", dataset.classes)
print("Class-to-index:", dataset.class_to_idx)

Output:

```
Number of images: 2188
Classes: ['paper', 'rock', 'scissors']
Class-to-index: {'paper': 0, 'rock': 1, 'scissors': 2}
```

`ImageFolder` sorts the class folders alphabetically and assigns each an integer starting at `0`, which is why the mapping runs paper, rock, scissors. The image count and the exact class names depend on the real dataset.

Read a single item to see the shape of one image tensor and its label. Indexing the dataset returns a pair: the image tensor and its integer class label.

In [ ]:
image, label = dataset[0]
print("Image shape:", image.shape)
print("Label:", label)

Output:

```
Image shape: torch.Size([3, 128, 128])
Label: 0
```

The shape is three channels, 128 pixels tall, and 128 pixels wide. The three channels are red, green, and blue: the loader converts every image to RGB, so the channel count is `3` regardless of the original file. The height and width are `128` because the `Resize` transform forced them.

To confirm the image loaded correctly, display it alongside its class name. This is a good point to verify visually that the label matches the pose in the picture.

In [ ]:
import matplotlib.pyplot as plt

image, label = dataset[0]
plt.imshow(image.permute(1, 2, 0))   # move channels last for display
plt.title(dataset.classes[label])
plt.axis("off")
plt.show()

`plt.imshow` expects the color channel as the last dimension, but the image tensor has channels first, so `permute(1, 2, 0)` reorders the dimensions from channels-height-width to height-width-channels for display only.

\[SCREENSHOT: the notebook output cell showing the rendered rock-paper-scissors image with its class name as the title. Capture the image and the title together so the learner can confirm the picture and its label agree.\]

## Step 7\. Reshape Images into Model-Ready Vectors

The first model in this course is built from linear layers, and a linear layer takes a flat vector rather than a three-dimensional grid. The final preparation step is flattening each image tensor into one long vector, the same `reshape(-1)` you practiced in Step 3\.

In [ ]:
image, label = dataset[0]
flat = image.reshape(-1)
print("Flat shape:", flat.shape)

Output:

```
Flat shape: torch.Size([49152])
```

The length `49152` is `3 × 128 × 128`, every pixel value laid end to end. When you process several images at once, you keep the first dimension for the batch and flatten the rest. The next block builds a small batch of four images by hand and reshapes it so each image becomes one row.

In [ ]:
images = torch.stack([dataset[i][0] for i in range(4)])   # shape: (4, 3, 128, 128)
batch = images.reshape(images.shape[0], -1)               # keep batch, flatten the rest
print("Batch of images:", images.shape)
print("Flattened batch:", batch.shape)

Output:

```
Batch of images: torch.Size([4, 3, 128, 128])
Flattened batch: torch.Size([4, 49152])
```

`torch.stack` combines the four individual image tensors into one tensor with a new leading dimension of size four, the batch dimension. Reshaping with `images.shape[0]` as the first dimension and `-1` for the rest keeps the four images separate while flattening each into a 49,152-value row. A batch shaped as rows of features is exactly what a linear layer accepts, which is where the course goes next. \[Inferred from docs: `torch.stack` and `reshape` are documented operations; this batch-then-flatten combination is applied for the lab\]

## Cleanup

This lab runs entirely in a notebook and creates no services or long-lived processes, so cleanup is light.

On Google Colab, the runtime is temporary. When you disconnect or the session ends, the installed packages and the cloned `LFD2000` folder are cleared automatically, and nothing else remains. No action is required.

On a local Jupyter environment, the only artifact on disk is the cloned dataset folder. You can remove it if you want to return to your pre-lab state, or keep it to save re-downloading.

In [ ]:
!rm -rf LFD2000

**Warning:** This permanently deletes the cloned dataset folder and everything in it. Run it only if you no longer need the local copy; the later chapters re-acquire the dataset in their own notebooks, so removing it now does not block anything.

Output: the `LFD2000` directory is gone; a subsequent `ls` no longer lists it. Cleanup is complete, and the environment returns to its pre-lab state. Nothing from this lab needs to persist for later chapters.